# Experimento A/B en página de inicio

El objetivo de este proyecto es evaluar un **experimento A/B** realizado en una página de inicio (landing page) con versiónes **A y B** para apoyar una **decisión de negocio basada en datos**.

---

El archivo `landing_experiment.csv` contiene información de usuarios expuestos a dos versiones de la página de inicio (landing page) dentro del experimento A/B. Incluye región, dispositivo, fuente de tráfico, tipo de usuario, conversión y gasto.

El análisis sigue una lógica clara y progresiva:

1. 🔍 Explorar y validar los datos.

2. 💰 Comparar el **gasto promedio** por usuario entre la página A y B.

3. 🎯 Comparar la **tasa de conversión** entre la página A y B.

4. 🌐 Revisar **la relación entre la fuente de tráfico y la conversión**.

5. 👤 Revisar **la relación entre el tipo de usuario y la conversión**.

6. 📈 **Visualizar los resultados**: Respalda tus conclusiones mediante gráficos claros.

Se aplican **puebas estadísticas apropiados** para comparar las páginas y **recomendar qué versión es mejor**, justificando la decisión con datos.

## 🧩 Paso 1: Cargar y validar los datos

### 1.1 Carga de datos y vista rápida

In [2]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
# cargar archivo
df = pd.read_csv('/datasets/landing_experiment.csv')

**Vista previa e información general del conjunto de datos**

In [4]:
# mostrar las primeras 5 filas
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         40000 non-null  object 
 1   date            40000 non-null  object 
 2   landing         40000 non-null  object 
 3   region          40000 non-null  object 
 4   dispositivo     40000 non-null  object 
 5   traffic_source  40000 non-null  object 
 6   user_type       40000 non-null  object 
 7   converted       40000 non-null  int64  
 8   gasto           40000 non-null  float64
dtypes: float64(1), int64(1), object(7)
memory usage: 2.7+ MB


In [5]:
# información general
df.head()

,user_id,date,landing,region,dispositivo,traffic_source,user_type,converted,gasto
0,26f3052e-8500-44ea-8fff-06de65258abb,2026-01-01,A,Norte,Mobile,Email,Recurrente,1,38.08
1,92378c09-4bbf-40c7-945e-82b84f392d22,2026-01-23,A,Occidente,Mobile,Organic,Nuevo,0,0.00
2,a4397360-40e5-45d6-a7ff-dcb4da2c9a1f,2026-01-01,B,Centro,Mobile,Organic,Nuevo,0,0.00
3,7ca3a26f-1e6c-44aa-9b09-b8cb01112956,2026-01-22,A,Centro,Mobile,Ads,Nuevo,0,0.00
4,8dc9593b-5b9c-479d-848b-a99493920419,2026-01-16,A,Sur,Mobile,Organic,Nuevo,0,0.00


In [6]:
#Correccion de formato de dato para la columna 'date'
df['date'] = pd.to_datetime(df['date'])

In [7]:
#Revision si hay datos nulos 
df.isna().sum()

user_id           0
date              0
landing           0
region            0
dispositivo       0
traffic_source    0
user_type         0
converted         0
gasto             0
dtype: int64

In [8]:
#revision si hay filas duplicadas
df.duplicated().sum()

0

In [9]:
#revision de los valores numericos de la columna 'gasto'
df['gasto'].describe()

count    40000.000000
mean         9.325554
std         25.667986
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        303.680000
Name: gasto, dtype: float64

In [10]:
# revisar posibles outliers in columna 'gasto'
df[df['gasto']>100].sort_values(by ='gasto',ascending=False)

,user_id,date,landing,region,dispositivo,traffic_source,user_type,converted,gasto
35068,029502f6-56c2-4e82-8e22-d7d2f3dd8515,2026-01-10,A,Sur,Desktop,Email,Nuevo,1,303.68
15189,2a4da7a1-d0db-41c7-b9ab-efd4c4748e0c,2026-01-02,A,Oriente,Desktop,Ads,Recurrente,1,269.91
10822,1ea885eb-8650-4c22-8cd2-e3718d4082af,2026-01-09,B,Centro,Mobile,Organic,Recurrente,1,249.99
1185,8771dabd-5b60-468e-915a-60ef6c0ebb66,2026-01-12,B,Norte,Desktop,Organic,Nuevo,1,244.89
22823,0035c847-2e8b-4499-be6c-b4c624f58e56,2026-01-10,B,Norte,Desktop,Ads,Nuevo,1,231.37
...,...,...,...,...,...,...,...,...,...
39313,e36bdb92-b9d0-412c-b430-5ddb11d29e90,2026-01-23,A,Sur,Desktop,Organic,Recurrente,1,100.17
14056,dac4e64b-e1e9-45f0-8071-2bfdb6fe2962,2026-01-27,B,Occidente,Desktop,Ads,Recurrente,1,100.15
711,cd5af5bf-692a-4058-b0a6-b5510934d311,2026-01-19,A,Norte,Mobile,Ads,Nuevo,1,100.12
34862,723e7e22-3316-44f5-b33b-04054de29b24,2026-01-18,B,Norte,Desktop,Ads,Recurrente,1,100.08


In [17]:
resultado = df[df['gasto']>100].sort_values(by='gasto', ascending=False)

In [18]:
resultado.head()

,user_id,date,landing,region,dispositivo,traffic_source,user_type,converted,gasto
35068,029502f6-56c2-4e82-8e22-d7d2f3dd8515,2026-01-10,A,Sur,Desktop,Email,Nuevo,1,303.68
15189,2a4da7a1-d0db-41c7-b9ab-efd4c4748e0c,2026-01-02,A,Oriente,Desktop,Ads,Recurrente,1,269.91
10822,1ea885eb-8650-4c22-8cd2-e3718d4082af,2026-01-09,B,Centro,Mobile,Organic,Recurrente,1,249.99
1185,8771dabd-5b60-468e-915a-60ef6c0ebb66,2026-01-12,B,Norte,Desktop,Organic,Nuevo,1,244.89
22823,0035c847-2e8b-4499-be6c-b4c624f58e56,2026-01-10,B,Norte,Desktop,Ads,Nuevo,1,231.37


✍️ **Comentario**: Haz doble clic en este bloque y escribe qué ves
- Se observa que la columna 'date' no tiene el formato correcto, ya que deberia ser 'datetime'.Convertimos la columna 'date' a formato 'datetime' para poder trabajar con las fechas correctas.
 -Se han revisado tambien los valores minimos y maximos de la columna de variables numericas 'gasto' y se ha encontrado que el valor maximo es mayor con $33 lo que es un salto parecido a los siguiente gastos mas altos.   


**Descripción del conjunto de datos**

El dataset contiene las siguientes columnas:

- `user_id` — Identificador único del usuario
- `date` — Fecha en la que el usuario fue expuesto a la página
- `landing` — Versión de la página mostrada al usuario
- `region` — Región geográfica del usuario
- `dispositivo` — Tipo de dispositivo utilizado por el usuario
- `traffic_source` — Canal por el que llegó el usuario
- `user_type` — Tipo de usuario según historial previo
- `converted` — Indica si el usuario realizó una conversión
- `gasto` — Monto gastado por el usuario (0 si no convirtió)

### 1.2 Análisis exploratorio y revisión de calidad de datos

Se identifican las variables clave del experimento A/B y se valida que estén bien definidas, completas y que sean consistentes.


 **Variable `user_id`**  
 Verificar usuarios únicos

In [19]:
#Revision de usuarios unicos 
df['user_id'].nunique()


40000

In [23]:
#Revision de usuarios unicos 
df.shape[0]

40000

 **Variable `date`**  
Explorar rango de fechas

In [21]:
# Resumen estadístico
df["date"].describe()

count                   40000
unique                     28
top       2026-01-24 00:00:00
freq                     1512
first     2026-01-01 00:00:00
last      2026-01-28 00:00:00
Name: date, dtype: object

In [22]:
# Identificar rango temporal del experimento
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

Fecha mínima: 2026-01-01 00:00:00
Fecha máxima: 2026-01-28 00:00:00


**Variable `gasto` (numérica)**

In [24]:
# Resumen estadístico
print(df['gasto'].describe())

count    40000.000000
mean         9.325554
std         25.667986
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        303.680000
Name: gasto, dtype: float64


In [25]:
df[df['converted']==1].shape

(5706, 9)

In [26]:
df[df['gasto']>0].shape

(5706, 9)

In [28]:
#Filtro por los usuarios convertidos para revision de resumen estadistico
usuarios_convertidos=df[df['converted']==1]
usuarios_convertidos['gasto'].describe()

count    5706.000000
mean       65.373668
std        30.896545
min        12.120000
25%        42.950000
50%        59.860000
75%        80.370000
max       303.680000
Name: gasto, dtype: float64

In [30]:
# Resumen estadístico de usuarios que se convirtieron
usuarios_convertidos.groupby('dispositivo')['gasto'].mean()

dispositivo
Desktop    66.861993
Mobile     64.259363
Name: gasto, dtype: float64

In [31]:
#conteo de usuarios convertidos por tipo de dispositivo
usuarios_convertidos.groupby('dispositivo')['gasto'].count()

dispositivo
Desktop    2443
Mobile     3263
Name: gasto, dtype: int64

 **Variables categóricas**  
 Verificar categorías esperadas del experimento ( A y B).

In [32]:
# Explorar variables categóricas y cómo se distribuyen
print("\nConteo de categorías:")
print(df[['region','dispositivo']].value_counts())


Conteo de categorías:
region     dispositivo
Norte      Mobile         6931
Centro     Mobile         6018
Sur        Mobile         5009
Norte      Desktop        4235
Occidente  Mobile         3915
Centro     Desktop        3595
Sur        Desktop        3030
Oriente    Mobile         2956
Occidente  Desktop        2483
Oriente    Desktop        1828
dtype: int64


In [34]:
#Calculo de porcentaje de conversion por region y dispositivo
pd.crosstab(df['region'], df['dispositivo'], normalize='index')*100

dispositivo,Desktop,Mobile
region,,
Centro,37.397275,62.602725
Norte,37.927637,62.072363
Occidente,38.809003,61.190997
Oriente,38.210702,61.789298
Sur,37.691255,62.308745


In [35]:
usuarios_convertidos.groupby('dispositivo')['gasto'].sum()

dispositivo
Desktop    163343.85
Mobile     209678.30
Name: gasto, dtype: float64

✍️ **Comentario**: 
-En columna 'user_id' se han encontrado solo usuarios unicos .
- En columna 'date' se observa que la informacion colectada es en el mes de Enero 2026 entre 1 y 28 de Enero, siendo el dia de 24 Enero , dia con la mayor cantidad de transacciones.
  -en columna'gasto' se han encontrado solo 5706 usuarios del total de 40000 que se han convertido a clientes. De los usuarios_convertidos se puede notar un gasto que tiene un minimo de 12 y un maximo de 303 con una media de 59,6 y un promedio de 65,3.
  -sobre las variables categoricas 'region','dispositivo' se ha observado que en todas las regiones hay el mismo patron de conversion (mobile>desktop). En termino de gasto realizado por tipo de dispositivo, se puede ver que hay un total mayor de gastos en mobile a pesar que el promedio de gasto por usuario es levemente mayor en desktop que en mobile.
  



## 💰 Paso 2: Comparar el gasto promedio por usuario (página A vs B)

Se evalua si existen diferencias estadísticamente significativas en el gasto promedio de los **usuarios que se convirtieron en clientes** entre la página A y la página B, para identificar qué versión genera **mayor valor económico** para el negocio.


In [ ]:
# Gasto por versión
gasto_A = #completa el código
gasto_B = #completa el código

# Verificar cantidad de datos que tiene cada grupo
len(gasto_A), len(gasto_B)

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Aplicar prueba



# Visualizar resultados
print(f"Estadístico : {...}")
print(f"Valor p: {...}")

### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica con tus propias palabras qué indican estos resultados sobre el gasto promedio entre la página A y la página B.


---


## 📈 Paso 3: Comparar la tasa de conversión entre la página A y B

Se evalua si existen difere]ncias estadísticamente significativas en la **tasa de conversión** entre la página A y B, con el fin de identificar qué versión genera **mayor número de usuarios convertidos**.

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Número de usuarios convertidos por página


# Total de usuarios por página


print("Usuarios convertidos por página:\n", ...)
print("\nTotal de usuarios por página:\n", ...)


In [ ]:
# Aplicar prueba



# Visualizar resultados
print(f"Estadístico : {...}")
print(f"Valor p: {...}")

### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica qué indica el resultado sobre la tasa de conversión entre la página A y la página B.

## 🔗 Paso 4: Revisar la relación entre la fuente de tráfico y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre la **fuente de tráfico** (`traffic_source`) y la **conversión** (`converted`), para identificar qué canales generan más conversiones.

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Aplicar prueba



### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica qué indican los resultados considerando tanto las cantidades absolutas como las tasas de conversión.


## 👤 Paso 5: Revisar la relación entre el tipo de usuario y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre el **tipo de usuario** (`user_type`) y la **conversión** (`converted`), entendiendo que un usuario recurrente puede haber visitado antes sin necesariamente convertirse en cliente en esta ocasión.

El objetivo es identificar qué perfiles muestran mayor probabilidad de conversión dentro del contexto analizado.

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Aplicar prueba



### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica qué indica el resultado.

## 📊 Paso 6: Visualizar los resultados de variables categóricas

Se explora visualmente la relación entre variables categóricas (`traffic_source` y `user_type`) y la conversión, mostrando para cada categoría:
- la cantidad absoluta de usuarios que convirtieron y no convirtieron,
- la proporción de usuarios que convirtieron y no convirtieron.

Esto permite analizar tanto el impacto en volumen como la efectividad relativa de cada categoría y reforzar los resultados obtenidos en las pruebas estadísticas.

### Relación entre la fuente de tráfico y la conversión

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

### Relación entre el tipo de usuario y la conversión

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

## 🧩 Paso 7. Insight Ejecutivo para Stakeholders

Se traducen los hallazgos del análisis del experimento A/B en conclusiones accionables para el negocio, enfocadas en **versión de página, conversión, gasto promedio, canales de tráfico y tipo de usuario**.

**Preguntas a responder:**  
- ¿Qué página genera mayor conversión y gasto promedio?  
- ¿Qué canales de tráfico son más efectivos para generar conversiones?  
- ¿Existen diferencias significativas según el tipo de usuario?  
- ¿Qué recomendaciones se pueden tomar para optimizar la estrategia de marketing?


---

### 🌟 Insight Ejecutivo basado en el Experimento A/B

#### 🔍 **Comparación de página (A vs B)**  

**Gasto promedio por usuario que convirtió:**
- Observacion 1 aquí
- Observacion 2 aquí
- **Interpretación:**

<br>

**Tasa de conversión:** 
- Observacion 1 aquí
- Observacion 2 aquí
- **Interpretación:**

---

#### 📊 **Segmentación por fuente de tráfico**
- Observacion aquí
- **Interpretación:**
 
 ---

#### 📊 **Segmentación por tipo de usuario**
- Observacion aquí
- **Interpretación:**

---

Las visualizaciones usadas respaldan los resultados estadísticos de pasos anteriores.

---

#### 💡 **Recomendaciones de negocio:** 
- Recomendación aquí
-  Recomendación aquí